In [149]:
# tensorflow pytorch를 중복으로 사용시 실행 허용
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

In [ ]:
from ultralytics import YOLO

model = YOLO('./pkl/yolov8s.pt')
model.names

In [151]:
# 폴더에서 opencv를 이용한 이미지 파일 읽기
import cv2
image_path = './image/bus_people.jpeg'
image_path2 = './image/bus_people2.jpg'

img = cv2.imread(image_path)
img2 = cv2.imread(image_path2)

img.shape, img2.shape

((343, 600, 3), (896, 1200, 3))

In [152]:
# TF (NWHC) => (1, 640, 448,   3)
# PT (NCHW) => (1, 3  , 384, 640)
result = model(img2)
result = result[0]
result


0: 480x640 10 persons, 29 cars, 3 motorcycles, 5 buss, 1 traffic light, 1 backpack, 127.8ms
Speed: 3.5ms preprocess, 127.8ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 640)


ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
keypoints: None
masks: None
names: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted plant',

In [153]:
class_ids = result.boxes.cls.cpu().numpy().astype(int)
class_ids

array([ 2,  2,  5,  2,  2,  2,  2,  2,  2,  2,  0,  2,  2,  2,  2,  2,  2,  5,  0,  0,  2,  2,  2,  5,  2,  3,  2,  9,  2,  3,  0,  5,  0,  0,  2,  2,  2,  2,  0, 24,  0,  2,  0,  3,  2,  2,  5,  0,  2])

In [154]:
# 신뢰도. 정확도하고는 다른개념임을 유의
# cpu().numpy().타입
confidences = result.boxes.conf.cpu().numpy()
confidences

array([    0.88634,     0.88129,     0.87809,     0.87298,     0.85293,     0.84338,     0.82779,     0.82324,     0.80957,     0.80806,     0.79433,     0.77076,     0.76216,     0.73992,     0.72331,     0.71329,      0.7082,     0.70681,     0.67833,     0.66465,     0.64933,     0.62512,     0.60316,     0.59511,
           0.58784,     0.58713,      0.5777,     0.55873,     0.55541,     0.54977,     0.53665,     0.52338,     0.52026,     0.51587,     0.48958,     0.46624,     0.44871,     0.43811,     0.40547,     0.38971,     0.36607,     0.35428,     0.34117,     0.31961,     0.31391,     0.30101,     0.28818,     0.28373,
           0.25904], dtype=float32)

In [155]:
for i, cls_id in enumerate(class_ids):
    print(f"{i + 1}, {model.names[int(cls_id)]}, {confidences[i]:.2f}")

1, car, 0.89
2, car, 0.88
3, bus, 0.88
4, car, 0.87
5, car, 0.85
6, car, 0.84
7, car, 0.83
8, car, 0.82
9, car, 0.81
10, car, 0.81
11, person, 0.79
12, car, 0.77
13, car, 0.76
14, car, 0.74
15, car, 0.72
16, car, 0.71
17, car, 0.71
18, bus, 0.71
19, person, 0.68
20, person, 0.66
21, car, 0.65
22, car, 0.63
23, car, 0.60
24, bus, 0.60
25, car, 0.59
26, motorcycle, 0.59
27, car, 0.58
28, traffic light, 0.56
29, car, 0.56
30, motorcycle, 0.55
31, person, 0.54
32, bus, 0.52
33, person, 0.52
34, person, 0.52
35, car, 0.49
36, car, 0.47
37, car, 0.45
38, car, 0.44
39, person, 0.41
40, backpack, 0.39
41, person, 0.37
42, car, 0.35
43, person, 0.34
44, motorcycle, 0.32
45, car, 0.31
46, car, 0.30
47, bus, 0.29
48, person, 0.28
49, car, 0.26


In [87]:
rendered_img = result.plot()

cv2.imshow("yolo", rendered_img)
cv2.waitKey(0)
cv2.destroyAllWindows()

---

In [184]:
# 신뢰도가 80이상 필터
target_image_path = image_path

filter_threshold = 0.8
target_image = cv2.imread(target_image_path)


result = model(target_image)[0]

filtered_boxes = [] # 박스 보관 리스트
names_table = model.names

for box in result.boxes:
    conf = float(box.conf.item())
    if conf < filter_threshold:
        continue
    cls_id = int(box.cls.item())
    name = names_table[cls_id]
    coords = map(int, box.xyxy[0])
    filtered_boxes.append((*coords, cls_id, conf, name))


0: 384x640 10 persons, 1 car, 1 bus, 1 traffic light, 2 backpacks, 86.0ms
Speed: 1.7ms preprocess, 86.0ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)


In [185]:
for fb in filtered_boxes:
    xmin, ymin, xmax, ymax, class_id, conf, name = fb
    print(f"Label: {name:8s} | "\
        f"Class id: {class_id} | " \
        f"Conf: {conf:.2f} | "\
        f"Box: ({xmin:4d}, {ymin:4d}, {xmax:4d}, {ymax:4d})")

Label: bus      | Class id: 5 | Conf: 0.93 | Box: ( 193,   25,  573,  323)
Label: car      | Class id: 2 | Conf: 0.87 | Box: ( 561,  170,  599,  206)
Label: person   | Class id: 0 | Conf: 0.86 | Box: ( 176,  158,  247,  341)
Label: person   | Class id: 0 | Conf: 0.86 | Box: (  69,  159,  117,  291)
Label: person   | Class id: 0 | Conf: 0.81 | Box: ( 121,  153,  155,  277)


In [186]:
for x1, y1, x2, y2, class_id, conf, name in filtered_boxes:
    label = f"{name}:{conf:.2f}"

    # cv2.rectangle(이미지, (x1, y1), (x2, y2), 색상, 두께)
    cv2.rectangle(target_image, (x1, y1), (x2, y2), (0, 255, 0), 1)

    # cv2.putText(이미지, 문자, (x, y위치), 폰트, 폰트크기, 색상, 두께)
    cv2.putText(target_image, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 255), 1)

cv2.imshow("yolo", target_image)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
import cv2

video_path = "./video/bus2.mp4"

cap = cv2.VideoCapture(video_path)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    result = model(frame)[0]
    rendered_frame = result.plot()
    cv2.imshow("yolo", rendered_frame)

    # ESC키 또는 q키를 누르면 종료
    key = cv2.waitKey(1) & 0xFF
    if key == 27 or key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

---

In [ ]:
import cv2

video_path = "./video/bus2.mp4"

cap = cv2.VideoCapture(video_path)
filter_threshold = 0.7
names_table = model.names

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    result = model.predict(frame)[0]
    for box in result.boxes:
        conf = float(box.conf.item())
        if conf < filter_threshold:
            continue
        name = names_table[int(box.cls.item())]
        (x1, y1, x2, y2) = map(int, box.xyxy[0])
        label = f"{name}:{conf:.2f}"
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 1)
        label_y1 = y1 - 10 if y1 > 20 else y1 + 10
        cv2.putText(frame, label, (x1, label_y1), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 255), 1)
        cv2.imshow("yolo", frame)

    # # ESC키 또는 q키를 누르면 종료
    key = cv2.waitKey(1) & 0xFF
    if key == 27 or key == ord('q'):
        break


cap.release()
cv2.destroyAllWindows()


0: 384x640 3 persons, 1 car, 1 bus, 1 train, 1 handbag, 1 cell phone, 102.6ms
Speed: 1.3ms preprocess, 102.6ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 1 car, 117.8ms
Speed: 1.6ms preprocess, 117.8ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 1 backpack, 1 handbag, 1 cell phone, 1 refrigerator, 115.8ms
Speed: 1.8ms preprocess, 115.8ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 3 handbags, 2 refrigerators, 109.4ms
Speed: 1.1ms preprocess, 109.4ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 1 suitcase, 1 refrigerator, 116.4ms
Speed: 1.3ms preprocess, 116.4ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 1 suitcase, 108.2ms
Speed: 1.2ms preprocess, 108.2ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 1 handbag, 1 suitcase, 124.

In [22]:
import cv2
import time

video_path = "./video/bus2.mp4"

cap = cv2.VideoCapture(video_path)
filter_threshold = 0.7
names_table = model.names

fps = cap.get(cv2.CAP_PROP_FPS) or 30

# ── 속도 옵션 (필요에 맞게 조절) ──────────────────────────
imgsz  = 320      # 추론 입력 크기 ↓ = 빠름 (기본 640)
device = None     # CUDA 있으면 0, 없으면 None(CPU)
# ────────────────────────────────────────────────────────

play_start = time.time()          # wall-clock anchor
frame_index = 0

while cap.isOpened():
    # grab(): 디코딩 없이 다음 프레임으로 이동 (싸다)
    if not cap.grab():
        break
    frame_index += 1

    # 이 프레임이 보여져야 할 시각보다 뒤처졌으면 → 디코딩/추론 없이 건너뛰기
    target_time = frame_index / fps
    if (time.time() - play_start) > target_time:
        continue

    # 실제로 표시할 프레임만 디코딩
    ret, frame = cap.retrieve()
    if not ret:
        break

    result = model.predict(frame, imgsz=imgsz, device=device, verbose=False)[0]
    for box in result.boxes:
        conf = float(box.conf.item())
        if conf < filter_threshold:
            continue
        name = names_table[int(box.cls.item())]
        (x1, y1, x2, y2) = map(int, box.xyxy[0])
        label = f"{name}:{conf:.2f}"
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 1)
        label_y1 = y1 - 10 if y1 > 20 else y1 + 10
        cv2.putText(frame, label, (x1, label_y1), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 255), 1)

    cv2.imshow("yolo", frame)

    # 다음 프레임 표시 시각까지 남은 시간만 대기
    wait_s = target_time - (time.time() - play_start)
    wait_ms = max(1, int(wait_s * 1000))
    key = cv2.waitKey(wait_ms) & 0xFF
    if key == 27 or key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

---

In [ ]:
model = YOLO('./pkl/yolov8s.pt')

# freeze => 숫자만큼 학습하지않고 고정
# 0 데이터가 많을때 전체를 파인튜닝
# 5 보통데이터
# 10 데이터가 적을때

model.train( 
    data = './data.yaml',
    epochs = 20,
    imgsz=640,
    batch=16,
    freeze=10,
    device="cpu",
    project="runs",
    name="yolov8_transfer"
)

In [38]:
# 이어서 학습
model = YOLO('./runs/detect/runs/yolov8_transfer/weights/best.pt')
model.names

{0: 'daisy', 1: 'dandelion', 2: 'roses', 3: 'sunflowers', 4: 'tulips'}

In [39]:
model.train( 
    data = './data.yaml',
    epochs=20,
    imgsz=640,
    batch=16,
    freeze=10,
    device="cpu",
    project="runs",
    name="exp1",
    exist_ok=True,
)

Ultralytics 8.4.60  Python-3.13.9 torch-2.12.0+cpu CPU (12th Gen Intel Core i7-1260P)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=./runs/detect/runs/yolov8_transfer/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=exp1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000018D89D42C80>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
    

In [49]:
import cv2

video_path = "./video/꽃 동영상.mp4"
model = YOLO('./runs/detect/runs/exp1/weights/best.pt')

cap = cv2.VideoCapture(video_path)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    result = model(frame)[0]
    rendered_frame = result.plot()
    cv2.imshow("yolo", rendered_frame)

    # ESC키 또는 q키를 누르면 종료
    key = cv2.waitKey(1) & 0xFF
    if key == 27 or key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


0: 480x640 2 daisys, 137.5ms
Speed: 2.1ms preprocess, 137.5ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 daisys, 128.9ms
Speed: 4.1ms preprocess, 128.9ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 daisy, 139.5ms
Speed: 2.7ms preprocess, 139.5ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 120.8ms
Speed: 2.1ms preprocess, 120.8ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 daisy, 145.9ms
Speed: 2.0ms preprocess, 145.9ms inference, 1.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 daisy, 144.6ms
Speed: 3.4ms preprocess, 144.6ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 daisy, 127.6ms
Speed: 2.3ms preprocess, 127.6ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 daisy, 130.6ms
Speed: 2.0ms preprocess, 130.6ms inference, 0.8ms postprocess per image 